In [1]:
import dataclasses, time
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import scipy.sparse as sp
from collections import namedtuple
import jax
import jax.numpy as jnp
import jax.scipy.linalg
from itertools import combinations

jax.config.update("jax_enable_x64", True)
from pyblock3.fcidump import FCIDUMP
from pyblock3.hamiltonian import Hamiltonian
from pyblock3.algebra.mpe import MPE
from pyblock3.algebra.symmetry import SZ

from trot.core.system import System
from trot.core.ops import k_energy
from trot.ham.hubbard import HamHubbard
from trot.ham.chol import HamChol
from trot.trial.uhf import UhfTrial, get_rdm1 as uhf_get_rdm1
from trot.trial.auto import make_auto_trial_ops
from trot.core.ops import MeasOps
from trot.meas.uhf import energy_kernel_uw_rh, build_meas_ctx as uhf_build_meas_ctx
from trot.prop import blocks
from trot.prop.cpmc import init_prop_state
from trot.prop.hubbard_cpmc_ops import _build_prop_ctx, make_hubbard_cpmc_ops
from trot.prop.types import PropOps, PropState
from trot import walkers as wk
from trot.walkers import _qr as _qr_t
from trot.prop.types import QmcParams
from trot.driver import run_qmc_energy

In [2]:
L = 16
n_up = 8
n_down = 8  
t = 1.0
U = 4.0

h1 = np.zeros((L, L))
for i in range(L - 1):
    h1[i, i + 1] = h1[i + 1, i] = -t

ham = HamHubbard(h1=jnp.asarray(h1), u=U)
sys_ = System(norb=L, nelec=(n_up, n_down), walker_kind="unrestricted")

eps, mo = np.linalg.eigh(h1)
Ca, Cb = mo[:, :n_up].copy(), mo[:, :n_down].copy()
e_hf = 2.0 * eps[:n_up].sum() + U * sum((Ca[i] @ Ca[i]) * (Cb[i] @ Cb[i]) for i in range(L))

# walkers are initialised from this
trial_data = UhfTrial(mo_coeff_a=jnp.asarray(Ca), mo_coeff_b=jnp.asarray(Cb))
def energy_uhf(walker, ham_data=None, meas_ctx=None, trial_data=trial_data):
    """Single-determinant Hubbard local energy, used only as a cross-check of the
    MPS/MSD estimators below."""
    return energy_kernel_uw_rh(walker, ham_chol, meas_ctx_uhf, trial_data)
# the Hubbard interaction is exactly Cholesky-decomposable -- g2e[i,i,i,i] = U gives
# one vector per site, L^g = sqrt(U) E_gg -- which is what meas/uhf's kernel wants.
chol = np.zeros((L, L, L))
for i in range(L):
    chol[i, i, i] = np.sqrt(U)
ham_chol = HamChol(h0=jnp.asarray(0.0), h1=jnp.asarray(h1),
                   chol=jnp.asarray(chol), basis="restricted")
meas_ctx_uhf = uhf_build_meas_ctx(ham_chol, trial_data)

print(f"E_HF             = {e_hf:.12f}")

E_HF             = -3.675902894919


In [3]:
#Full ED benchmark, and the determinant basis the MSD route runs in. Est runtime 20s

# One string list serves both spins: the MSD kernels index alpha and beta with the
# same ROWS, which is only right at n_up == n_down.
assert n_up == n_down, "the MSD bookkeeping below indexes both spins with one string list"

STR = [frozenset(c) for c in combinations(range(L), n_up)]
IDX = {s: k for k, s in enumerate(STR)}
NS = len(STR)
OCCA = np.array([[1 if i in s else 0 for i in range(L)] for s in STR])
ROWS = [np.array(sorted(s)) for s in STR]


def _hop_matrix():
    """Spinless nearest-neighbour hopping in the string basis, SPARSE.

    The Jordan-Wigner string between ADJACENT sites is empty, so every nonzero
    element is just -t and there is no sign to track. Sparse because it is 0.062%
    nonzero at L=16 half filling: 804 KB against 1264 MB dense, and it takes
    `apply_H` from 17.6 s to 1.9 s. Duplicate (row, col) entries are summed on
    construction, which is what the `+=` did.
    """
    rows, cols, vals = [], [], []
    for k, s in enumerate(STR):
        for i in range(L - 1):
            for src, dst in ((i + 1, i), (i, i + 1)):
                if src in s and dst not in s:
                    rows.append(IDX[frozenset((s - {src}) | {dst})])
                    cols.append(k); vals.append(-t)
    return sp.csr_matrix((vals, (rows, cols)), shape=(NS, NS))


HOP = _hop_matrix()
# |r_a ^ r_b|, the double occupancy of determinant (a, b) -- one matmul, not an
# NS^2 python loop: 1.1 s against 31 s at L=16 half filling, and int8 rather than
# float64 keeps it at 158 MB instead of 1.3 GB.
DOCC = (OCCA.astype(np.int64) @ OCCA.astype(np.int64).T).astype(np.int8)


def apply_H(X):
    """H X in the determinant basis, X[a, b] indexed by (alpha string, beta string).

    H = sum_sigma sum_ij h1_ij c+_i c_j + U sum_i n_ia n_ib is one-body within each
    spin channel plus a diagonal, so applying it is two NS x NS matrix products and
    an elementwise scaling:

        (H X)[a,b] = sum_a' HOP[a,a'] X[a',b] + sum_b' HOP[b,b'] X[a,b'] + U d_ab X[a,b]

    No two-electron integrals and no FCI library. This is the MSD route's
    Hamiltonian; it shares no code with the MPO the MPS route uses, which is what
    makes comparing the two routes worth anything.
    """
    return HOP @ X + (HOP @ X.T).T + U * DOCC * X


def ed_energy():
    """Ground state by dense eigh in the (n_up, n_down) sector."""
    HOPd = HOP.toarray()                    # HOP is sparse; this needs it dense
    H = np.zeros((NS * NS, NS * NS))
    Hr = H.reshape(NS, NS, NS, NS)          # a view, so these write into H
    for ib in range(NS):
        Hr[:, ib, :, ib] += HOPd            # hopping, spin up
    for ia in range(NS):
        Hr[ia, :, ia, :] += HOPd            # hopping, spin down
    H[np.diag_indices_from(H)] += U * DOCC.ravel()
    return float(np.linalg.eigvalsh(H)[0])


# e_exact = ed_energy()
# print(f"E_exact           = {e_exact:.12f}")

In [4]:
def plan_channel_maxB(C):
    """Fishman-White plan with B = n-k always: the remaining block is a genuine
    projector, so its eigenvalues are exactly 0/1 for any walker."""
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []
    for k in range(n - 1):
        B = n - k
        w, W = np.linalg.eigh((U @ U.T)[k : k + B, k : k + B])
        v, occ[k] = (W[:, 0], 0) if w[0] <= 1.0 - w[-1] else (W[:, -1], 1)
        Bs.append(B); vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])
    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs


def plan_channel(C, eps_occ=1e-10):
    """Fishman-White plan with B grown only until a mode isolates.

    The block is enlarged until some eigenvalue of Lambda_B comes within eps_occ
    of 0 or 1. For a low-entangled state that happens at small B, so both the
    gate count and the bond dimension stay far below the maximal-B plan -- which
    is what makes L > 16 reachable at all, since maximal B gives chi = 2^(L/2)
    (1024 at L = 20, 4096 at L = 24).

    The price: the retained block is only APPROXIMATELY idempotent, so the
    replay must use mode="eigh". See `channel_angles`.
    """
    U, n = np.asarray(C, float).copy(), C.shape[0]
    occ, Bs, vrefs = np.zeros(n, int), [], []
    for k in range(n - 1):
        Lam = U @ U.T
        for B in range(2, n - k + 1):                  # grow until a mode isolates
            w, W = np.linalg.eigh(Lam[k : k + B, k : k + B])
            if min(w[0], 1.0 - w[-1]) < eps_occ:
                break
        v, occ[k] = (W[:, 0], 0) if w[0] <= 1.0 - w[-1] else (W[:, -1], 1)
        Bs.append(B); vrefs.append(v.copy())
        for j in range(B - 1, 0, -1):
            th = np.arctan2(v[j], v[j - 1])
            c, s = np.cos(th), np.sin(th)
            v[j - 1], v[j] = c * v[j - 1] + s * v[j], 0.0
            U[k + j - 1], U[k + j] = (c * U[k + j - 1] + s * U[k + j],
                                      -s * U[k + j - 1] + c * U[k + j])
    occ[n - 1] = int(round(float(U[n - 1] @ U[n - 1])))
    assert occ.sum() == C.shape[1], "particle number lost during compression"
    return occ, np.array(Bs), vrefs



def channel_angles(C, plan, xp=jnp):
    """Replay the Givens angles from C with the plan frozen. C must be ORTHONORMAL.   
    """
    occ, Bs, vrefs = plan
    C = xp.asarray(C)
    rows = [C[i] for i in range(C.shape[0])]
    angles = []
    for k, (B, vref) in enumerate(zip(Bs, vrefs)):
        B = int(B)
        Uk = xp.stack(rows[k : k + B])
        M = Uk @ Uk.T
        vr = xp.asarray(vref)
        _, W = xp.linalg.eigh(M)
        v = W[:, -1] if occ[k] == 1 else W[:, 0]
        v = v * xp.sign(v @ vr)
        v = [v[i] for i in range(B)]
        for j in range(B - 1, 0, -1):
            th = xp.arctan2(v[j], v[j - 1])
            c, s = xp.cos(th), xp.sin(th)
            v[j - 1] = c * v[j - 1] + s * v[j]
            p = k + j - 1
            rows[p], rows[p + 1] = (c * rows[p] + s * rows[p + 1],
                                    -s * rows[p] + c * rows[p + 1])
            angles.append((p, th))
    return angles, rows


def V_hat(th):
    """The two-site gate, as a (2,2,2,2) tensor. The reference definition;
    `_gate_pair` fuses it into the contraction."""
    c, s = jnp.cos(th), jnp.sin(th)
    g = jnp.eye(4).at[1, 1].set(c).at[1, 2].set(s).at[2, 1].set(-s).at[2, 2].set(c)
    return g.reshape(2, 2, 2, 2)


def _gate_pair(A, B, th, xp=jnp):
    """(A B) with the gate folded in, as (Dl, 2, 2, Dr)."""
    c, s = xp.cos(th), xp.sin(th)
    t00 = A[:, 0, :] @ B[:, 0, :]
    t01 = A[:, 0, :] @ B[:, 1, :]
    t10 = A[:, 1, :] @ B[:, 0, :]
    t11 = A[:, 1, :] @ B[:, 1, :]
    return xp.stack([xp.stack([t00, c * t01 + s * t10], 1),
                     xp.stack([-s * t01 + c * t10, t11], 1)], 1)


def sector_plan(ql, qr):
    """Static per-charge row/column index sets for one two-site split, plus the
    middle labels and the gather maps back to full shape. NumPy, hence cached."""
    nl, nr = len(ql), len(qr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()
    secs, qm, rcat, ccat = [], [], [], []
    for nm in sorted(set(rc.tolist()) & set(cc.tolist())):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        k = min(len(r), len(c))
        secs.append((r, c, k)); qm += [nm] * k
        rcat.append(r); ccat.append(c)
    rcat, ccat = np.concatenate(rcat), np.concatenate(ccat)
    rmap = np.full(2 * nl, len(rcat), int); rmap[rcat] = np.arange(len(rcat))
    cmap = np.full(2 * nr, len(ccat), int); cmap[ccat] = np.arange(len(ccat))
    return secs, np.array(qm, int), rmap, cmap


_SEC_CACHE = {}

def _sectors(ql, qr):
    key = (ql.tobytes(), len(ql), qr.tobytes(), len(qr))
    if key not in _SEC_CACHE:
        _SEC_CACHE[key] = sector_plan(ql, qr)
    return _SEC_CACHE[key]


def split_full(T, ql, qr):
    """Exact split, full rank in each particle-number sector, via QR."""
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors(ql, qr)
    As, Bs = [], []
    for r, c, k in secs:
        q, rr = jnp.linalg.qr(M[np.ix_(r, c)], mode="reduced")
        As.append(q[:, :k]); Bs.append(rr[:k])
    A = jax.scipy.linalg.block_diag(*As)
    B = jax.scipy.linalg.block_diag(*Bs)
    A = jnp.concatenate([A, jnp.zeros((1, A.shape[1]), A.dtype)], 0)[rmap]
    B = jnp.concatenate([B, jnp.zeros((B.shape[0], 1), B.dtype)], 1)[:, cmap]
    return A.reshape(Dl, 2, -1), B.reshape(-1, 2, Dr), qm




def channel_mps(C, plan):
    """Product state |occ> -> gates in reverse derivation order -> one d=2 MPS.

    Returns (tensors, bond labels, gauge sign). This notebook fixes the gauge by
    the amplitude ratio in Part 3 instead, so the third value is unused here.
    """
    one_hot = (jnp.array([[[1.0], [0.0]]]), jnp.array([[[0.0], [1.0]]]))

    occ = plan[0]
    ts = [one_hot[int(o)] for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, rows = channel_angles(C, plan)
    for p, th in reversed(angles):
        ts[p], ts[p + 1], qn[p + 1] = split_full(_gate_pair(ts[p], ts[p + 1], th),
                                                 qn[p], qn[p + 2])
    gauge = jnp.linalg.det(jnp.stack([rows[i] for i in np.where(occ == 1)[0]]))
    return ts, qn, gauge


def combine(Aa, qna, Ab, qnb):
    """Interleave two d=2 channels into one d=4 MPS, local index n_a + 2 n_b."""
    ts, qn = [], [np.zeros((1, 2), int)]
    for i in range(len(Aa)):
        Dal, _, Dar = Aa[i].shape
        Dbl, _, Dbr = Ab[i].shape
        out = jnp.zeros((Dal, Dbl, 4, Dar, Dbr))
        for na in (0, 1):
            sgn = (-1.0) ** (na * qnb[i])                 # the Jordan-Wigner sign
            for nb in (0, 1):
                out = out.at[:, :, na + 2 * nb, :, :].set(
                    jnp.einsum("ar,b,bs->abrs", Aa[i][:, na, :], sgn, Ab[i][:, nb, :]))
        ts.append(out.reshape(Dal * Dbl, 4, Dar * Dbr))
        qn.append(np.stack([np.repeat(qna[i + 1], Dbr), np.tile(qnb[i + 1], Dar)], 1))
    return ts, qn


def sd_to_mps_qn(ca, cb, plan_a, plan_b):
    """Slater determinant -> (d=4 MPS, per-bond (n_a, n_b) labels). ORTHONORMAL ca/cb."""
    ta, qna, _ = channel_mps(ca, plan_a)
    tb, qnb, _ = channel_mps(cb, plan_b)
    return combine(ta, qna, tb, qnb)


def sd_to_mps(ca, cb, plan_a, plan_b):

    return sd_to_mps_qn(ca, cb, plan_a, plan_b)[0]


def mps_overlap(bra, ket):
    """<bra|ket>, dense. The charge-blocked version is in Part 3."""
    e = jnp.ones((bra[0].shape[0], ket[0].shape[0]))
    for a, b in zip(bra, ket):
        e = jnp.tensordot(a, jnp.tensordot(e, b, ([1], [0])), ([0, 1], [0, 1]))
    return e.reshape(())


def hubbard_mpo(L, t, U):
    """Dw=6 MPO. Bond basis: 0 = nothing started, 1..4 = a hop is pending, 5 = done."""
    I4 = np.eye(4)
    cr_a = np.zeros((4, 4)); cr_a[1, 0] = 1.0; cr_a[3, 2] = 1.0
    cr_b = np.zeros((4, 4)); cr_b[2, 0] = 1.0; cr_b[3, 1] = -1.0
    an_a, an_b = cr_a.T.copy(), cr_b.T.copy()
    n_a, n_b = np.diag([0., 1., 0., 1.]), np.diag([0., 0., 1., 1.])
    P_a, P_b = np.diag([1., -1., 1., -1.]), np.diag([1., 1., -1., -1.])
    W = np.zeros((L, 6, 4, 4, 6))
    for i in range(L):
        W[i, 0, :, :, 0] = I4
        W[i, 5, :, :, 5] = I4
        W[i, 0, :, :, 5] = U * (n_a @ n_b)
        if i < L - 1:
            W[i, 0, :, :, 1] = cr_a @ P_b
            W[i, 0, :, :, 2] = an_a @ P_b
            W[i, 0, :, :, 3] = P_a @ cr_b
            W[i, 0, :, :, 4] = P_a @ an_b
        if i > 0:
            W[i, 1, :, :, 5] = -t * an_a
            W[i, 2, :, :, 5] = -t * cr_a
            W[i, 3, :, :, 5] = -t * an_b
            W[i, 4, :, :, 5] = -t * cr_b
    return W


def apply_mpo(W, ts):
    """(W psi)[i] has bond dimension Dw*chi; the boundary MPO bonds are projected out."""
    out = []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
    return out


def compress(ts, tol=1e-13):
    """Left-to-right QR to canonicalise, then right-to-left SVD. Exact at this tol."""
    ts = [np.asarray(t) for t in ts]
    for i in range(len(ts) - 1):
        Dl, d, Dr = ts[i].shape
        q, r = np.linalg.qr(ts[i].reshape(Dl * d, Dr))
        ts[i] = q.reshape(Dl, d, -1)
        ts[i + 1] = np.tensordot(r, ts[i + 1], axes=([1], [0]))
    for i in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[i].shape
        u, sv, vt = np.linalg.svd(ts[i].reshape(Dl, d * Dr), full_matrices=False)
        k = max(int((sv > tol * max(sv[0], 1e-300)).sum()), 1)
        ts[i] = vt[:k].reshape(k, d, Dr)
        ts[i - 1] = np.tensordot(ts[i - 1], u[:, :k] * sv[:k], axes=([2], [0]))
    return ts

In [5]:
def build_hamil(L, U, t=1.0):
    h1e = np.zeros((L, L))
    for i in range(L - 1):
        h1e[i, i + 1] = h1e[i + 1, i] = -t
    g2e = np.zeros((L,) * 4)
    for i in range(L):
        g2e[i, i, i, i] = U
    fd = FCIDUMP(pg="c1", n_sites=L, n_elec=n_up + n_down, twos=n_up - n_down,
                 ipg=0, h1e=h1e, g2e=g2e)
    return Hamiltonian(fd, flat=True)


def run_dmrg(hamil, bdim, n_sweeps=14, seed=0):
    
    np.random.seed(seed)    
    mpo, _ = hamil.build_qc_mpo().compress(cutoff=1e-12)
    mps = hamil.build_mps(bdim)
    dmrg = MPE(mps, mpo, mps).dmrg(bdims=[bdim] * n_sweeps, noises=[1e-5] * 6 + [0],
                                   dav_thrds=[1e-10], iprint=-1, n_sweeps=n_sweeps)
    return mps, float(dmrg.energies[-1])


def spin_occ(q):
    """(n_alpha, n_beta) of an SZ label: n = na + nb, 2Sz = na - nb."""
    return (int(q.n) + int(q.twos)) // 2, (int(q.n) - int(q.twos)) // 2


def flat_blocks(mps, i):
    """(q_labels, shape, data) for every block of site i, with Python-int labels."""
    t = mps[i]
    for k in range(t.n_blocks):
        q = tuple(SZ.from_flat(int(x)) for x in t.q_labels[k])
        sh = tuple(int(x) for x in t.shapes[k])
        yield q, sh, np.asarray(t.data[t.idxs[k]:t.idxs[k + 1]]).reshape(sh)


def densify(mps, L):
    """flat pyblock3 MPS -> dense (Dl, 4, Dr) arrays, local index n_a + 2 n_b."""
    qkey = lambda q: (int(q.n), int(q.twos))
    left, right = [], []
    for i in range(L):
        lo, ro = {}, {}
        for (ql, qp, qr), sh, _ in flat_blocks(mps, i):
            lo[qkey(ql)], ro[qkey(qr)] = sh[0], sh[2]
        left.append(lo); right.append(ro)
    for i in range(L - 1):
        assert left[i + 1] == right[i], f"bond {i+1} mismatch between sites"

    offs = []
    for b in [left[i] for i in range(L)] + [right[L - 1]]:
        o, acc = {}, 0
        for k in sorted(b):
            o[k] = (acc, b[k]); acc += b[k]
        offs.append((o, acc))

    out = []
    for i in range(L):
        A = np.zeros((offs[i][1], 4, offs[i + 1][1]))
        for (ql, qp, qr), sh, dat in flat_blocks(mps, i):
            assert sh[1] == 1, f"physical block dim {sh[1]} != 1"
            na, nb = spin_occ(qp)
            ol, dl = offs[i][0][qkey(ql)]
            orr, dr = offs[i + 1][0][qkey(qr)]
            A[ol:ol + dl, na + 2 * nb, orr:orr + dr] = dat[:, 0, :]
        out.append(A)
    return out


def bond_qns(mps, L):
    """Per-bond (n_a, n_b) label for every index of the densified MPS.
    The same offset bookkeeping `densify` uses, so the labels and the dense
    arrays are guaranteed to agree index for index. This is what the
    charge-blocked contraction in Part 3 needs.
    """
    qkey = lambda q: (int(q.n), int(q.twos))
    left, right = [], []
    for i in range(L):
        lo, ro = {}, {}
        for (ql, qp, qr), sh, _ in flat_blocks(mps, i):
            lo[qkey(ql)], ro[qkey(qr)] = sh[0], sh[2]
        left.append(lo); right.append(ro)
    out = []
    for b in [left[i] for i in range(L)] + [right[L - 1]]:
        lab = []
        for k in sorted(b):
            lab += [((k[0] + k[1]) // 2, (k[0] - k[1]) // 2)] * b[k]   # (n,2Sz)->(na,nb)
        out.append(np.array(lab, int))
    return out

def mps_amp(ts, occ_a, occ_b):
    """<d|MPS>, a bond-dimension-1 contraction."""
    v = np.ones((1, 1))
    for A, a, b in zip(ts, occ_a, occ_b):
        v = v @ np.asarray(A)[:, int(a) + 2 * int(b), :]
    return float(v[0, 0])


def interleaving_sign(ra, rb):
    """(-1)^K relating 'all alpha then all beta' to the interleaved lattice order."""
    return (-1.0) ** sum(int((rb < i).sum()) for i in ra)

CHI = 64                  
hamil = build_hamil(L, U)
mps_dmrg, E_dmrg_dav = run_dmrg(hamil, CHI)

LOCAL = {spin_occ(SZ.from_flat(int(c))): k for k, c in enumerate(hamil.basis[0])}
ket_T_np = densify(mps_dmrg, L)
ket_T = [jnp.asarray(t) for t in ket_T_np]

print(f"DMRG chi={CHI}:  bond dims {mps_dmrg.show_bond_dims()}")
print(f"local index map read from hamil.basis: {LOCAL}   -> l = n_a + 2 n_b")
print(f"dense bond dims : {[t.shape[0] for t in ket_T] + [ket_T[-1].shape[-1]]}")
print(f"<psi_T|psi_T>   = {float(mps_overlap(ket_T, ket_T)):.12f}")

#COnverting the trial in the det basis and applying H to compare later.
def interleaving_sign_all(occ):
    """(-1)^K for EVERY (a, b) pair at once.

    K = sum_{i in r_a} #{j in r_b : j < i} is bilinear in the occupation vectors,
    K[a,b] = occ_a . LOW . occ_b with LOW[i,j] = 1 iff j < i, so the whole NS x NS
    sign matrix is two matmuls instead of NS^2 python calls. Bit-identical to
    `interleaving_sign`; at L=16 half filling it is 1.4 s against ~20 minutes,
    and 158 MB as int8 against 1.2 GB as float64.
    """
    n = occ.shape[1]
    o = occ.astype(np.int64)
    LOW = np.tril(np.ones((n, n), np.int64), -1)
    return (1 - 2 * ((o @ LOW @ o.T) & 1)).astype(np.int8)


def mps_amp_all(ts, occ, chunk=1 << 20):
    """<d|MPS> for every (alpha, beta) pair, as an (NS, NS) array.

    The same contraction as `mps_amp`, but all four local states are contracted
    and THEN selected, so each site is one GEMM over a batch of determinants
    rather than NS^2 python-level calls -- 1.8 us per amplitude against 24 us,
    and the two agree to 3e-13. Chunked so the intermediate stays bounded; the
    output itself is NS^2 floats and that is irreducible (1.2 GB at L=16 half
    filling), which is the MSD route being exponential, as advertised.
    """
    ns = occ.shape[0]
    A = [jnp.asarray(t) for t in ts]
    flat = np.empty(ns * ns)
    for lo in range(0, ns * ns, chunk):
        hi = min(lo + chunk, ns * ns)
        ia, ib = np.divmod(np.arange(lo, hi), ns)
        cfg = jnp.asarray(occ[ia] + 2 * occ[ib])
        v = jnp.ones((hi - lo, 1))
        for x, Ax in enumerate(A):
            w = jnp.tensordot(v, Ax, axes=([1], [0]))          # (N, 4, chi')
            v = jnp.take_along_axis(w, cfg[:, x][:, None, None], axis=1)[:, 0, :]
        flat[lo:hi] = np.asarray(v[:, 0])
    return flat.reshape(ns, ns)


ISG = interleaving_sign_all(OCCA)
AMP = mps_amp_all(ket_T_np, OCCA)
M = ISG * AMP
print(f"|c_d| > 1e-10 : {(np.abs(AMP) > 1e-10).sum()} of {NS * NS} determinants")

MH = apply_H(M)                                        
Hket_T = [jnp.asarray(t) for t in compress(apply_mpo(hubbard_mpo(L, t, U), ket_T_np))]

e_T = float((M * MH).sum() / (M * M).sum())
print(f"H|psi_T> bond dims : {[t.shape[0] for t in Hket_T] + [Hket_T[-1].shape[-1]]}")
print(f"<psi_T|H|psi_T>    = {e_T:.12f}")
# print(f"E_exact            = {e_exact:.12f}")
# print(f"trial error        = {e_T - e_exact:+.3e}")

#Defining overlap and mixed exp values functions for the MSD approach
Mj, MHj = jnp.asarray(M), jnp.asarray(MH)
RA = jnp.asarray(np.stack(ROWS))                     

def _dets(c):
    return jax.vmap(lambda r: jnp.linalg.det(c[r, :]))(RA)


def msd_overlap(walker, trial_data=None):
    ca, cb = walker
    return _dets(ca) @ Mj @ _dets(cb)


def msd_energy(walker, ham_data=None, meas_ctx=None, trial_data=None):
    ca, cb = walker
    da, db = _dets(ca), _dets(cb)
    return (da @ MHj @ db) / (da @ Mj @ db)

QC MPO site   0 / 16
QC MPO site   1 / 16
QC MPO site   2 / 16
QC MPO site   3 / 16
QC MPO site   4 / 16
QC MPO site   5 / 16
QC MPO site   6 / 16
QC MPO site   7 / 16
QC MPO site   8 / 16
QC MPO site   9 / 16
QC MPO site  10 / 16
QC MPO site  11 / 16
QC MPO site  12 / 16
QC MPO site  13 / 16
QC MPO site  14 / 16
QC MPO site  15 / 16
DMRG chi=64:  bond dims 1|4|16|64|64|64|64|64|64|64|64|64|64|64|16|4|1
local index map read from hamil.basis: {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}   -> l = n_a + 2 n_b
dense bond dims : [1, 4, 16, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 16, 4, 1]
<psi_T|psi_T>   = 1.000000000000
|c_d| > 1e-10 : 116516629 of 165636900 determinants
H|psi_T> bond dims : [1, 4, 16, 64, 244, 313, 318, 315, 318, 315, 318, 313, 244, 64, 16, 4, 1]
<psi_T|H|psi_T>    = -8.818756381246


In [6]:
plan_a  = plan_channel(Ca)
plan_b =  plan_channel(Cb)

ROWS_A, ROWS_B = np.where(plan_a[0] == 1)[0], np.where(plan_b[0] == 1)[0]
ISIGN_REF = interleaving_sign(ROWS_A, ROWS_B)
LOC_REF = [int(a) + 2 * int(b) for a, b in zip(plan_a[0], plan_b[0])]


def amp_exact_ref(ca, cb):
    return ISIGN_REF * jnp.linalg.det(ca[ROWS_A, :]) * jnp.linalg.det(cb[ROWS_B, :])


def amp_mps_ref(ts):
    v = jnp.ones((1, 1))
    for A, l in zip(ts, LOC_REF):
        v = v @ A[:, l, :]
    return v[0, 0]


def _bra(walker):
    ca, cb = walker
    qa, _ = jnp.linalg.qr(ca)
    qb, _ = jnp.linalg.qr(cb)
    return sd_to_mps(qa, qb, plan_a, plan_b), ca, cb


def mps_overlap_T(walker, trial_data=None):
    bra, ca, cb = _bra(walker)
    return (amp_exact_ref(ca, cb) / amp_mps_ref(bra)) * mps_overlap(bra, ket_T)


def mps_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    bra, _, _ = _bra(walker)
    return mps_overlap(bra, Hket_T) / mps_overlap(bra, ket_T)

---
### The charge-blocked overlap

Both the walker MPS and the DMRG trial conserve $(n_\alpha, n_\beta)$, so the
environment is nonzero only between bond indices carrying the *same* charge.
Exploiting that with sparse kernels would not port to a GPU, so instead each
bond is sorted by charge and each sector padded to a rectangle: the environment
becomes one dense array `E[q, a, b]` and a site update becomes two **batched
GEMMs** plus a segment-sum, with every shape static. The padding is zero, so the
result is exact rather than an approximation.

In [7]:
"""The charge-blocked overlap: dense, batched, GPU-shaped.

Both MPSs conserve (n_alpha, n_beta), so the environment E[c, r] is nonzero only
where the bra bond index c and the ket bond index r carry the SAME charge. The
point is to use that WITHOUT sparse kernels, which do not port to a GPU: sort
every bond by charge, pad each charge sector to a common size, and carry the
environment as one dense array

    E[q, a, b]        q = charge, a = bra index within q, b = ket index within q

A site update is then two BATCHED GEMMs and one segment-sum, every shape static,
no data-dependent indexing:

    Ein[t] = E[src[t]]                           gather, static indices
    tmp[t] = bra_blk[t]^T Ein[t]                 batched GEMM
    out[t] = tmp[t] ket_blk[t]                   batched GEMM
    E'[q'] = sum_{t : dst[t] = q'} out[t]        segment-sum

`t` runs over the allowed (charge in, physical index, charge out) transitions,
fixed by the bond labels, so the whole layout is planned once in NumPy -- the
same plan/replay split used for the two-site splits.

Padding is zero and the padded entries of the blocks are zero, so this is EXACT:
it returns the same number as the dense contraction, not an approximation.

Two things make it pay here beyond the charge blocking itself:

  * only charges present on BOTH sides survive. A bra charge the ket does not
    have can never reach the last bond, so all work on it is dropped. With a
    chi=8 trial against a chi=256 walker that removes most of the walker.
  * the ket is FIXED, so its padded blocks are extracted once, in NumPy, and
    there is no runtime gather on that side.
"""
PHYS_NAB = np.array([[l % 2, l // 2] for l in range(4)])      # l -> (n_a, n_b)


def _charge_index(qn):
    """{(na,nb): array of bond indices} for one bond's label array."""
    out = {}
    for i, q in enumerate(map(tuple, np.asarray(qn).tolist())):
        out.setdefault(q, []).append(i)
    return {k: np.array(v, int) for k, v in out.items()}


def block_plan(qn_bra, qn_ket):
    """Static layout and transition table for every site. All NumPy."""
    n = len(qn_bra) - 1
    bidx = [_charge_index(q) for q in qn_bra]
    kidx = [_charge_index(q) for q in qn_ket]
    charges, A, B = [], [], []
    for x in range(n + 1):
        q = sorted(set(bidx[x]) & set(kidx[x]))
        charges.append(q)
        A.append(max([len(bidx[x][c]) for c in q], default=0))
        B.append(max([len(kidx[x][c]) for c in q], default=0))

    sites = []
    for x in range(n):
        pos_in = {c: i for i, c in enumerate(charges[x])}
        pos_out = {c: i for i, c in enumerate(charges[x + 1])}
        src, dst, ri, ci, mb, lid = [], [], [], [], [], []
        for c in charges[x]:
            rows = bidx[x][c]
            for l in range(4):
                c2 = (c[0] + PHYS_NAB[l][0], c[1] + PHYS_NAB[l][1])
                if c2 not in pos_out:
                    continue
                cols = bidx[x + 1][c2]
                R = np.zeros((A[x], A[x + 1]), int)      # padded index grids; the
                C = np.zeros((A[x], A[x + 1]), int)      # padding points at 0 and
                M = np.zeros((A[x], A[x + 1]))           # is masked to zero
                R[:len(rows), :len(cols)] = rows[:, None]
                C[:len(rows), :len(cols)] = cols[None, :]
                M[:len(rows), :len(cols)] = 1.0
                src.append(pos_in[c]); dst.append(pos_out[c2])
                ri.append(R); ci.append(C); mb.append(M); lid.append(l)
        sites.append(dict(src=np.array(src, int), dst=np.array(dst, int),
                          ri=np.stack(ri), ci=np.stack(ci), mb=np.stack(mb),
                          lid=np.array(lid, int), nq_out=len(charges[x + 1])))
    return dict(sites=sites, charges=charges, A=A, B=B, bidx=bidx, kidx=kidx, n=n)


def ket_blocks(ket, plan):
    """Pre-extract the fixed side's padded blocks once: no runtime gather there."""
    out = []
    for x, st in enumerate(plan["sites"]):
        blk = np.zeros((len(st["src"]), plan["B"][x], plan["B"][x + 1]))
        for t, (qi, qo, l) in enumerate(zip(st["src"], st["dst"], st["lid"])):
            c, c2 = plan["charges"][x][qi], plan["charges"][x + 1][qo]
            rows, cols = plan["kidx"][x][c], plan["kidx"][x + 1][c2]
            blk[t, :len(rows), :len(cols)] = \
                np.asarray(ket[x])[np.ix_(rows, [l], cols)][:, 0, :]
        out.append(jnp.asarray(blk))
    return out


def blocked_overlap(bra, kblk, plan):
    """<bra|ket> with the charge structure carried as dense padded blocks."""
    E = jnp.ones((1, plan["A"][0], plan["B"][0]))
    for x, (st, kb) in enumerate(zip(plan["sites"], kblk)):
        bb = bra[x][st["ri"], st["lid"][:, None, None], st["ci"]] * st["mb"]
        Ein = E[st["src"]]                                  # (T, A_x,   B_x)
        tmp = jnp.einsum("tij,tik->tjk", bb, Ein)           # (T, A_x+1, B_x)
        out = jnp.einsum("tjk,tkl->tjl", tmp, kb)           # (T, A_x+1, B_x+1)
        E = jax.ops.segment_sum(out, st["dst"], num_segments=st["nq_out"])
    return E.reshape(())


def plan_report(plan):
    """stored environment entries: padded blocks vs exact blocks vs dense."""
    pad = sum(len(plan["charges"][x]) * plan["A"][x] * plan["B"][x]
              for x in range(plan["n"] + 1))
    exact = sum(sum(len(plan["bidx"][x][c]) * len(plan["kidx"][x][c])
                    for c in plan["charges"][x]) for x in range(plan["n"] + 1))
    dense = sum(sum(len(v) for v in plan["bidx"][x].values())
                * sum(len(v) for v in plan["kidx"][x].values())
                for x in range(plan["n"] + 1))
    return dict(dense=dense, padded_blocks=pad, exact_blocks=exact,
                transitions=sum(len(st["src"]) for st in plan["sites"]))



qn_ket = bond_qns(mps_dmrg, L)

In [8]:
"""Two knobs, and a gauge fix that survives them.

WALKER COMPRESSION (CHI_WALKER) caps the CHANNEL bond dimension of the walker
conversion. Carried over from `mps_trial_cpmc.ipynb`: truncation is a discrete
decision, so it goes in the plan -- a NumPy dry run records how many states each
charge sector of each split keeps, which keeps shapes static under vmap while the
singular values are still recomputed from every walker. The splits never need an
SVD of the big matrix; following arXiv:2212.09782, reduce with a QR and read the
Schmidt values off the small hermitian R R^T,

    M = Q R,   R R^T = V S^2 V^T,   M_k = (Q V_k) (V_k^T R)

which is the exact rank-k truncation at about a third of an SVD's cost.

Unlike everything else here this one is an APPROXIMATION: it makes the walker MPS
inexact, so the MPS and MSD routes stop agreeing to machine precision.
CHI_WALKER = None (the default) keeps the conversion exact and the agreement at
1e-14. Note chi_channel is at most 16 at L=8 half filling, so CHI_WALKER >= 16 is
no truncation at all.

THE GAUGE HAD TO CHANGE FOR IT TO BE USABLE. Part 3 fixes the scale between the
walker MPS and the true determinant by matching ONE reference amplitude,

    amp_exact_ref(ca, cb) / amp_mps_ref(bra)

which is exact for an exact MPS -- any amplitude would do -- but divides by a
single truncated number, so it amplifies truncation error violently. The robust
alternative uses no amplitude at all: the conversion is orthogonal up to a sign,

    <MPS|SD(q)> = det(U_rot[occ, :]) = +-1      and     |SD(C)> = det(R) |SD(Q)>

so  g = det(R_a) det(R_b) g_a g_b  with g_sigma handed back by `channel_mps_c`.
Measured against the MSD overlap on the perturbed batch:

    CHI_WALKER   discarded   ratio gauge   det gauge
      None        0.0e+00      3.2e-14      3.1e-15
      8           5.5e-04      6.3e+02      2.6e-01
      4           5.7e-01          inf       9.0e-01

The det gauge is better even in the exact case, and ~2400x better at chi=8. The
residual error at chi=8 is much larger than `discarded` because the bond plan is
frozen on the HF reference while the walkers have drifted away from it.
"""

CHI_WALKER = 4            # None = exact; int caps the CHANNEL bond dimension
CUTOFF_WALKER = 0.0          # additionally drop s < CUTOFF * s_max per split

BondPlan = namedtuple("BondPlan", "ks qn chi discarded")


def sector_plan_ks(ql, qr, ks=None):
    """sector_plan, keeping only ks[s] states in charge sector s."""
    nl, nr = len(ql), len(qr)
    rc = (ql[:, None] + np.arange(2)[None, :]).ravel()
    cc = (qr[None, :] - np.arange(2)[:, None]).ravel()
    secs, qm, rcat, ccat = [], [], [], []
    for s, nm in enumerate(sorted(set(rc.tolist()) & set(cc.tolist()))):
        r, c = np.where(rc == nm)[0], np.where(cc == nm)[0]
        k = min(len(r), len(c)) if ks is None else int(ks[s])
        if k == 0:
            continue
        secs.append((r, c, k)); qm += [nm] * k
        rcat.append(r); ccat.append(c)
    rcat, ccat = np.concatenate(rcat), np.concatenate(ccat)
    rmap = np.full(2 * nl, len(rcat), int); rmap[rcat] = np.arange(len(rcat))
    cmap = np.full(2 * nr, len(ccat), int); cmap[ccat] = np.arange(len(ccat))
    return secs, np.array(qm, int), rmap, cmap


_SEC_KS_CACHE = {}

def _sectors_ks(ql, qr, ks):
    key = (ql.tobytes(), len(ql), qr.tobytes(), len(qr), ks)
    if key not in _SEC_KS_CACHE:
        _SEC_KS_CACHE[key] = sector_plan_ks(ql, qr, ks)
    return _SEC_KS_CACHE[key]


def split_trunc(T, ql, qr, ks):
    """Compressing split, exact rank-k per sector, via QR + eigh of the small R R^T."""
    Dl, _, _, Dr = T.shape
    M = T.reshape(Dl * 2, 2 * Dr)
    secs, qm, rmap, cmap = _sectors_ks(ql, qr, tuple(ks))
    As, Bs = [], []
    for r, c, k in secs:
        q, rr = jnp.linalg.qr(M[np.ix_(r, c)], mode="reduced")
        _, V = jnp.linalg.eigh(rr @ rr.T)
        Vk = V[:, ::-1][:, :k]
        As.append(q @ Vk); Bs.append(Vk.T @ rr)
    A = jax.scipy.linalg.block_diag(*As)
    B = jax.scipy.linalg.block_diag(*Bs)
    A = jnp.concatenate([A, jnp.zeros((1, A.shape[1]), A.dtype)], 0)[rmap]
    B = jnp.concatenate([B, jnp.zeros((B.shape[0], 1), B.dtype)], 1)[:, cmap]
    return A.reshape(Dl, 2, -1), B.reshape(-1, 2, Dr), qm


def plan_bonds(C, plan, chi_max=None, cutoff=0.0):
    """NumPy dry run: freeze how each split spends its bond budget."""
    occ = plan[0]
    ts = [np.eye(2)[int(o)].reshape(1, 2, 1) for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, _ = channel_angles(np.asarray(C, float), plan, xp=np)

    ks_all, discarded = [], 0.0
    for p, th in reversed(angles):
        T = _gate_pair(ts[p], ts[p + 1], th, xp=np)
        Dl, _, _, Dr = T.shape
        M = T.reshape(Dl * 2, 2 * Dr)
        secs, _, _, _ = sector_plan_ks(qn[p], qn[p + 2])
        svs, blocks = [], []
        for r, c, kfull in secs:
            u, sv, vt = np.linalg.svd(M[np.ix_(r, c)], full_matrices=False)
            svs.append(sv[:kfull]); blocks.append((u, sv, vt))
        flat = np.concatenate(svs)
        order = np.argsort(-flat)
        sel = order[: len(flat) if chi_max is None else min(int(chi_max), len(flat))]
        if cutoff > 0.0 and len(flat):
            sel = sel[flat[sel] > cutoff * flat[order[0]]]
        keep = np.zeros(len(flat), bool); keep[sel] = True
        discarded += float((flat[~keep] ** 2).sum())
        ks, off = [], 0
        for sv in svs:
            ks.append(int(keep[off:off + len(sv)].sum())); off += len(sv)
        ks_all.append(tuple(ks))

        _, qm, rmap, cmap = sector_plan_ks(qn[p], qn[p + 2], tuple(ks))
        As, Bs = [], []
        for (u, sv, vt), k in zip(blocks, ks):
            if k:
                As.append(u[:, :k]); Bs.append(sv[:k, None] * vt[:k])
        A = scipy.linalg.block_diag(*As); B = scipy.linalg.block_diag(*Bs)
        A = np.concatenate([A, np.zeros((1, A.shape[1]))], 0)[rmap]
        B = np.concatenate([B, np.zeros((B.shape[0], 1))], 1)[:, cmap]
        ts[p] = A.reshape(Dl, 2, -1); ts[p + 1] = B.reshape(-1, 2, Dr)
        qn[p + 1] = qm
    return BondPlan(ks=ks_all, qn=qn, chi=max(len(q) for q in qn),
                    discarded=discarded)


def channel_mps_c(C, plan, bond_plan=None):
    """channel_mps with optional compression. Returns (tensors, labels, gauge sign)."""
    
    one_hot = (jnp.array([[[1.0], [0.0]]]), jnp.array([[[0.0], [1.0]]]))
    occ = plan[0]
    ts = [one_hot[int(o)] for o in occ]
    qn = [np.zeros(1, int)]
    for o in occ:
        qn.append(qn[-1] + int(o))
    angles, rows = channel_angles(C, plan)
    for gi, (p, th) in enumerate(reversed(angles)):
        T = _gate_pair(ts[p], ts[p + 1], th)
        if bond_plan is None:
            ts[p], ts[p + 1], qn[p + 1] = split_full(T, qn[p], qn[p + 2])
        else:
            ts[p], ts[p + 1], qn[p + 1] = split_trunc(T, qn[p], qn[p + 2],
                                                      bond_plan.ks[gi])
    return ts, qn, jnp.linalg.det(jnp.stack([rows[i] for i in np.where(occ == 1)[0]]))


# ------------------------------------------------- the walker conversion we use
bp_walk_a = bp_walk_b = None
if CHI_WALKER is not None or CUTOFF_WALKER > 0.0:
    bp_walk_a = plan_bonds(Ca, plan_a, chi_max=CHI_WALKER, cutoff=CUTOFF_WALKER)
    bp_walk_b = plan_bonds(Cb, plan_b, chi_max=CHI_WALKER, cutoff=CUTOFF_WALKER)


def convert_walker(ca, cb):
    """(d=4 tensors, bond labels, gauge) for one walker. No reference amplitude."""
    qa, ra = _qr_t(ca)
    qb, rb = _qr_t(cb)
    ta, qna, ga = channel_mps_c(qa, plan_a, bp_walk_a)
    tb, qnb, gb = channel_mps_c(qb, plan_b, bp_walk_b)
    ts, qn = combine(ta, qna, tb, qnb)
    return ts, qn, ra * rb * ga * gb


# the block plan has to be rebuilt: truncation changes the walker's bond labels
_bra_ref, qn_bra, _ = convert_walker(jnp.asarray(Ca), jnp.asarray(Cb))
blk_plan = block_plan(qn_bra, qn_ket)
ket_T_blk = ket_blocks(ket_T_np, blk_plan)

print("bond dims      walker", [len(q) for q in qn_bra])
print("               trial ", [len(q) for q in qn_ket])
print("charges/bond   walker", [len(set(map(tuple, q.tolist()))) for q in qn_bra])
print("               trial ", [len(set(map(tuple, q.tolist()))) for q in qn_ket])
print("               shared", [len(c) for c in blk_plan["charges"]],
      "  <- the chi=8 trial is what limits this")
print("padded block   bra   ", blk_plan["A"])
print("               ket   ", blk_plan["B"])
_rep = plan_report(blk_plan)
print(f"""
environment entries   dense {_rep['dense']}
                      charge blocks {_rep['exact_blocks']}
                      padded to rectangles {_rep['padded_blocks']}"""
      f"  ({_rep['dense'] / _rep['padded_blocks']:.1f}x fewer than dense)")
print(f"batched transitions per sweep: {_rep['transitions']}")



def blocked_overlap_T(walker, trial_data=None):
    """<psi_T|SD(C)>, charge-blocked, with the det gauge."""
    ca, cb = walker
    bra, _, gc = convert_walker(ca, cb)
    return gc * blocked_overlap(bra, ket_T_blk, blk_plan)

bond dims      walker [1, 4, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 16, 4, 1]
               trial  [1, 4, 16, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 64, 16, 4, 1]
charges/bond   walker [1, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 1]
               trial  [1, 4, 9, 16, 19, 14, 19, 14, 19, 14, 19, 14, 19, 16, 9, 4, 1]
               shared [1, 4, 9, 9, 9, 8, 9, 8, 9, 8, 9, 8, 9, 9, 9, 4, 1]   <- the chi=8 trial is what limits this
padded block   bra    [1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1]
               ket    [1, 1, 4, 9, 12, 10, 11, 10, 11, 10, 11, 10, 12, 9, 4, 1, 1]

environment entries   dense 11810
                      charge blocks 1216
                      padded to rectangles 4278  (2.8x fewer than dense)
batched transitions per sweep: 332


In [9]:
"""Blocking the ENERGY too: what it takes, and why it is off by default.

The overlap could be blocked because both sides carry (n_a, n_b) labels. The
energy needs <phi|H|psi_T>, and `Hket_T = compress(apply_mpo(W, ket_T))` has no
labels. They are lost in two distinct places:

  * `apply_mpo` -- RECOVERABLE. The output bond is the flattened pair (MPO bond
    dw, MPS bond c), so its charge is mpo_q[dw] + mps_q[c], and the Dw=6 Hubbard
    MPO's bond charges follow from how `hubbard_mpo` builds it: states 0
    ("nothing started") and 5 ("done") carry (0,0); state 1 is entered by cr_a so
    it carries (+1,0); state 2 by an_a -> (-1,0); 3 by cr_b -> (0,+1); 4 by an_b
    -> (0,-1). `apply_mpo_qn` below just carries that through, and the assertion
    checks every nonzero obeys q_left + q_phys = q_right.

  * `compress` -- ACTUALLY DESTROYS THEM. It QRs and SVDs the dense reshaped
    matrices with no charge sorting: the returned basis comes back in
    singular-value order, which interleaves charges arbitrarily; a degenerate
    singular value lets the factorisation MIX vectors of different charge; and
    the rank cut `(sv > tol*sv[0]).sum()` is one global count, so it can keep a
    partial sector. A charge-aware compress (sort by charge, factorise per
    sector, truncate per sector) would fix it -- that is the `sector_plan`
    machinery again, on a d=4 chain.

The way out taken here is simpler: SKIP `compress`. It is exact at tol=1e-13
anyway, so the uncompressed H|psi_T> is the same state, just at bond dimension
6*chi = 48 instead of 38. Verified below on random determinant amplitudes.

AND IT STILL DOES NOT PAY. Measured below: the blocked energy is exact to ~1e-14
but SLOWER than the dense one. The H side has 17 shared charges at the middle
bond with very uneven sector sizes, so padding every sector to a rectangle wastes
7378/1782 = 4.1x, which cancels most of the 11x block sparsity, and the extra
transitions cost more XLA ops. Fixing it needs size-bucketed padding (pad within
groups of similar-sized sectors instead of to one global max), which trades flops
for op count -- a bad trade at L=8, where these kernels are launch-bound.

It is also nearly irrelevant: the propagation needs the overlap at every one of
the 2L+2 = 18 field decisions per step against ONE energy evaluation per block.
At 20 prop steps that is 360 against 1, and the fast sweep serves all 360 from
one conversion plus a site loop. So the energy is ~1% of the work either
way. BLOCK_ENERGY is left False; the machinery is here and verified so the
measurement can be repeated rather than re-argued.
"""
BLOCK_ENERGY = False

# entering MPO bond state k has created this much charge
MPO_QN = np.array([[0, 0], [1, 0], [-1, 0], [0, 1], [0, -1], [0, 0]])


def apply_mpo_qn(W, ts, qn):
    """apply_mpo, carrying the bond labels.

    `apply_mpo` flattens the left bond as a*cl + c with the MPO index major, so a
    combined index has charge mpo_q[a] + mps_q[c].
    """
    out, qout = [], []
    for i, (w, A) in enumerate(zip(W, ts)):
        A = np.asarray(A)
        mq_l = MPO_QN[0:1] if i == 0 else MPO_QN
        if i == 0:
            w = w[0:1]
        if i == len(ts) - 1:
            w = w[:, :, :, 5:6]
        T = np.einsum("apqb,cqd->acpbd", w, A)
        dl, cl, p, dr, cr = T.shape
        out.append(T.reshape(dl * cl, p, dr * cr))
        qout.append(np.concatenate([mq_l[a][None, :] + qn[i] for a in range(dl)], 0))
    qout.append(MPO_QN[5][None, :] + qn[len(ts)])
    return out, qout


Hts_qn, Hqn = apply_mpo_qn(hubbard_mpo(L, t, U), ket_T_np, qn_ket)

_bad = 0
for _x, _A in enumerate(Hts_qn):
    for _l in range(4):
        _dq = np.array([_l % 2, _l // 2])
        for _r, _c in np.argwhere(np.abs(_A[:, _l, :]) > 1e-12):
            _bad += not np.array_equal(Hqn[_x][_r] + _dq, Hqn[_x + 1][_c])
print(f"H|psi_T>  uncompressed dims {[t.shape[0] for t in Hts_qn] + [1]}")
print(f"          compressed   dims {[t.shape[0] for t in Hket_T] + [1]}")
print(f"          charge-violating nonzeros: {_bad}   (must be 0)")

_rng2 = np.random.default_rng(1)
_e = 0.0
for _ in range(20):
    _oa = np.zeros(L, int); _oa[_rng2.choice(L, n_up, replace=False)] = 1
    _ob = np.zeros(L, int); _ob[_rng2.choice(L, n_down, replace=False)] = 1
    _e = max(_e, abs(mps_amp(Hts_qn, _oa, _ob)
                     - mps_amp([np.asarray(t) for t in Hket_T], _oa, _ob)))
print(f"          max |amp(uncompressed) - amp(compressed)| = {_e:.2e}  (same state)")

H_plan = block_plan(qn_bra, Hqn)
Hket_blk = ket_blocks(Hts_qn, H_plan)
_repH = plan_report(H_plan)
print(f"\nH-side env entries: dense {_repH['dense']}, blocks {_repH['exact_blocks']},"
      f" padded {_repH['padded_blocks']}"
      f"   -> {_repH['dense']/_repH['exact_blocks']:.1f}x sparsity but"
      f" {_repH['padded_blocks']/_repH['exact_blocks']:.1f}x padding waste")


def blocked_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc with BOTH contractions charge-blocked."""
    ca, cb = walker
    bra, _, _ = convert_walker(ca, cb)
    return (blocked_overlap(bra, Hket_blk, H_plan)
            / blocked_overlap(bra, ket_T_blk, blk_plan))


def dense_energy_T(walker, ham_data=None, meas_ctx=None, trial_data=None):
    """E_loc through the dense contraction against the compressed H|psi_T>."""
    ca, cb = walker
    bra, _, _ = convert_walker(ca, cb)
    return mps_overlap(bra, Hket_T) / mps_overlap(bra, ket_T)

energy_T = blocked_energy_T if BLOCK_ENERGY else dense_energy_T

H|psi_T>  uncompressed dims [1, 24, 96, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 384, 96, 24, 1]
          compressed   dims [1, 4, 16, 64, 244, 313, 318, 315, 318, 315, 318, 313, 244, 64, 16, 4, 1]
          charge-violating nonzeros: 0   (must be 0)
          max |amp(uncompressed) - amp(compressed)| = 3.67e-18  (same state)

H-side env entries: dense 70850, blocks 6001, padded 18970   -> 11.8x sparsity but 3.2x padding waste


In [10]:
"""THE FAST SWEEP, charge-blocked.

Reconverting the walker to an MPS and recontracting the overlap once per field
trial costs 2L+2 = 18 contractions per propagation step. This does ONE per step and then walks the
site loop with environments, which is where nearly all of the speed comes from.
It carries over from `mps_trial_cpmc.ipynb` unchanged in spirit -- it never used
the spin factorisation, only the fact that the discrete Hubbard-Stratonovich
operator is DIAGONAL in the local basis, so it applies to a spin-entangled DMRG
trial exactly as well.

Two reuses, both inside the blocked representation:

  * one pass of batched GEMMs per site serves both the field marginal and the
    pushed-forward left environment, because both need the same intermediate
        Q[t] = bra_blk[t]^T Lenv[src[t]] ket_blk[t]
    from which  M[l] = sum_{t: l(t)=l} <Q[t], R[x+1][dst[t]]>  and
                Lenv'[q'] = sum_{t: dst[t]=q'} D[l(t)] Q[t].
  * the right environments are built once per walker, backwards, and stay valid:
    sites already passed are folded into Lenv and sites still ahead are untouched.

The gauge prefactor is constant through the sweep, so it is computed once up
front: `convert_walker` returns det(R_a) det(R_b) g_a g_b, which fixes the scale
between the MPS and the true determinant without reference to any amplitude, and
the field operator is diagonal so it scales both descriptions identically.
"""

def right_envs_blocked(bra, kblk, plan):
    """R[x][q, a, b] = contraction of sites x..n-1, charge-blocked.

    R[0][0, 0, 0] is the full overlap, so the pre-loop value comes out free.
    """
    R = [jnp.ones((1, plan["A"][plan["n"]], plan["B"][plan["n"]]))]
    for x in range(plan["n"] - 1, -1, -1):
        st, kb = plan["sites"][x], kblk[x]
        bb = bra[x][st["ri"], st["lid"][:, None, None], st["ci"]] * st["mb"]
        Rin = R[-1][st["dst"]]                                # (T, A_x+1, B_x+1)
        tmp = jnp.einsum("tij,tjk->tik", bb, Rin)             # (T, A_x,   B_x+1)
        out = jnp.einsum("tik,tlk->til", tmp, kb)             # (T, A_x,   B_x)
        R.append(jax.ops.segment_sum(out, st["src"],
                                     num_segments=len(plan["charges"][x])))
    return R[::-1]


def fast_sweep(ca, cb, rns, hs, w_floor):
    """One walker's whole site loop, off a single conversion.

    Returns (ca, cb, overlap before the loop, overlap after, weight factor, nodes).
    """
    bra, _, pref = convert_walker(ca, cb)
    R = right_envs_blocked(bra, ket_T_blk, blk_plan)
    ov_in = pref * R[0][0, 0, 0]

    Lenv = jnp.ones((1, blk_plan["A"][0], blk_plan["B"][0]))
    ov, logw = ov_in, jnp.zeros(())
    nodes = jnp.zeros((), jnp.int32)
    # the two field choices as diagonal one-site operators, l = n_a + 2 n_b
    D0 = jnp.array([1.0, hs[0, 0], hs[0, 1], hs[0, 0] * hs[0, 1]])
    D1 = jnp.array([1.0, hs[1, 0], hs[1, 1], hs[1, 0] * hs[1, 1]])

    for x in range(L):
        st, kb = blk_plan["sites"][x], ket_T_blk[x]
        bb = bra[x][st["ri"], st["lid"][:, None, None], st["ci"]] * st["mb"]
        P = jnp.einsum("tij,tik->tjk", bb, Lenv[st["src"]])    # (T, A_x+1, B_x)
        Q = jnp.einsum("tjk,tkl->tjl", P, kb)                  # (T, A_x+1, B_x+1)

        # the field marginal, per local index, from the same Q
        w = jnp.einsum("tjl,tjl->t", Q, R[x + 1][st["dst"]])   # (T,)
        M = jax.ops.segment_sum(w, st["lid"], num_segments=4)

        ov0, ov1 = pref * (D0 @ M), pref * (D1 @ M)
        r0 = jnp.where(0.5 * (ov0 / ov) < w_floor, 0.0, 0.5 * (ov0 / ov))
        r1 = jnp.where(0.5 * (ov1 / ov) < w_floor, 0.0, 0.5 * (ov1 / ov))
        nodes = nodes + (r0 <= 0.0) + (r1 <= 0.0)        # the constrained-path test
        norm = r0 + r1 + 1.0e-13
        take0 = rns[x] < r0 / norm
        D = jnp.where(take0, D0, D1)
        ov = jnp.where(take0, ov0, ov1)
        logw = logw + jnp.log(norm)
        ca = ca.at[x, :].mul(jnp.where(take0, hs[0, 0], hs[1, 0]))
        cb = cb.at[x, :].mul(jnp.where(take0, hs[0, 1], hs[1, 1]))
        Lenv = jax.ops.segment_sum(D[st["lid"]][:, None, None] * Q, st["dst"],
                                   num_segments=len(blk_plan["charges"][x + 1]))
    return ca, cb, ov_in, ov, jnp.exp(logw), nodes


def init_prop_state_pinned(**kwargs):
    """trot's initializer with a strongly typed node counter.

    `trot.prop.cpmc.init_prop_state` sets node_encounters = jnp.asarray(0), which
    is WEAKLY typed, while every propagation step hands it back strongly typed.
    The aval of the state therefore differs between the first and second call of
    the jitted run_blocks, so the whole block scan is traced and compiled TWICE.
    Pinning the dtype costs one compile instead of two. Nothing under trot/ is
    modified.
    """
    return init_prop_state(**kwargs)._replace(
        node_encounters=jnp.zeros((), dtype=int))


def make_fast_prop_ops(ham_data, walker_kind, overlap_fn):
    """PropOps using the sweep instead of 2L+2 reconversions.

    `overlap_fn` is used for the two one-body half steps, so a run is consistent
    end to end with whichever estimator it is given. Same RNG stream, same weight
    clamps and same population control as trot's reference CPMC step, so two runs
    that differ only in their estimators stay comparable step for step rather
    than merely statistically.
    """
    cpmc_ops = make_hubbard_cpmc_ops(ham_data, walker_kind)

    def step(state, *, params, ham_data, trial_data, trial_ops,
             meas_ops, meas_ctx, prop_ctx):
        key, subkey = jax.random.split(state.rng_key)
        nw = wk.n_walkers(state.walkers)
        rns = jax.random.uniform(subkey, (nw, cpmc_ops.n_sites()))
        w_floor = float(getattr(params, "weight_floor", 1.0e-8))
        w_cap = float(getattr(params, "weight_cap", 100.0))
        damping = float(getattr(params, "pop_control_damping", 0.1))

        # --- first one-body half step, then the whole site loop on one conversion ---
        walkers = cpmc_ops.apply_one_body_half(state.walkers, prop_ctx)
        ca, cb, ov_half, overlaps, wfac, nod = jax.vmap(
            fast_sweep, in_axes=(0, 0, 0, None, None))(
            walkers[0], walkers[1], rns, prop_ctx.hs_constant, w_floor)

        ratio = jnp.real(jnp.real(ov_half) / state.overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = jnp.sum(ratio <= 0.0) + jnp.sum(nod)
        weights = jnp.where(state.weights * ratio > w_cap, 0.0, state.weights * ratio)
        weights = weights * wfac
        walkers = (ca, cb)

        # --- second one-body half step: a general rotation, so a full overlap ---
        walkers = cpmc_ops.apply_one_body_half(walkers, prop_ctx)
        overlaps_new = jnp.real(
            jax.vmap(overlap_fn, in_axes=(0, None))(walkers, trial_data))
        ratio = jnp.real(overlaps_new / overlaps)
        ratio = jnp.where(ratio < w_floor, 0.0, ratio)
        nodes = nodes + jnp.sum(ratio <= 0.0)
        weights = jnp.where(weights * ratio > w_cap, 0.0, weights * ratio)

        weights = weights * jnp.exp(prop_ctx.dt * state.pop_control_ene_shift)
        weights = jnp.where(weights > w_cap, 0.0, weights)
        avg_w = jnp.clip(jnp.mean(weights), min=1.0e-300)
        return PropState(
            walkers=walkers, weights=weights, overlaps=overlaps_new, rng_key=key,
            pop_control_ene_shift=state.e_estimate
            - damping * (jnp.log(avg_w) / prop_ctx.dt),
            e_estimate=state.e_estimate,
            node_encounters=state.node_encounters + nodes,
        )

    return PropOps(init_prop_state=init_prop_state_pinned,
                   build_prop_ctx=lambda h, t, p: _build_prop_ctx(h, p.dt),
                   step=step)

params = QmcParams(dt=0.01, n_walkers=32, n_prop_steps=20, n_blocks=40,
                   n_eql_blocks=20, weight_floor=1e-8, seed=1234)


def run(overlap_fn, energy_fn, prop_ops):
    """trial_data is used only to initialise walkers; the trial itself is baked
    into the closures above. `prop_ops` is always a fast-sweep propagator, built
    by `make_fast_prop_ops` below for this same overlap."""
    trial_ops = make_auto_trial_ops(sys_, overlap_u=overlap_fn, get_rdm1=uhf_get_rdm1)
    meas_ops = MeasOps(overlap=overlap_fn, kernels={k_energy: energy_fn})
    return run_qmc_energy(sys=sys_, params=params, ham_data=ham, trial_data=trial_data,
                          meas_ops=meas_ops, trial_ops=trial_ops, prop_ops=prop_ops,
                          block_fn=blocks.block)
                          
prop_ops_fast = make_fast_prop_ops(ham, sys_.walker_kind, blocked_overlap_T)
mean_mps, err_mps, be_mps, bw_mps = run(blocked_overlap_T, energy_T, prop_ops_fast)



Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/20]   -8.8191765927  3.200000e+01           0       0.0
[eql    4/20]   -8.8188056038  3.382042e+01           0      39.3
[eql    8/20]   -8.8186613069  3.359673e+01           0      41.6
[eql   12/20]   -8.8186260865  3.327908e+01           0      43.9
[eql   16/20]   -8.8187422255  3.148347e+01           3      46.2
[eql   20/20]   -8.8187795418  3.240536e+01          12      48.5

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    4/40]   -8.8187931352                 -8.8187931352  3.232184e+01          16     12.737      50.9
[blk    8/40]   -8.8188061733                 -8.8188200270  3.041897e+01          23      0.596      53.3
[blk   12/40]   -8.8188098518   3.138e-05     -8.8188171914  3.144494e+01          25      0.582      55.7
[blk   16/40]   -8.8187939939   2.561e-05     -8.8187463830  3.137074e+01       

In [11]:
prop_ops_msd = make_fast_prop_ops(ham, sys_.walker_kind, msd_overlap)
_t0 = time.time()
mean_msd, err_msd, be_msd, bw_msd = run(msd_overlap, msd_energy, prop_ops_msd)
t_msd = time.time() - _t0


Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/20]   -8.8190908900  3.200000e+01           0       0.0


/Users/fnappi/.trot/lib/python3.12/site-packages/jax/_src/interpreters/mlir.py:1286: UserWarning: A large amount of constants were captured during lowering (2.65GB total). If this is intentional, disable this warning by setting JAX_CAPTURED_CONSTANTS_WARN_BYTES=-1. To obtain a report of where these constants were encountered, set JAX_CAPTURED_CONSTANTS_REPORT_FRAMES=-1.
  warnings.warn(message)


[eql    4/20]   -8.8188360011  3.313498e+01           0      62.4
[eql    8/20]   -8.8188833399  3.213000e+01           0     110.1
[eql   12/20]   -8.8188233675  3.137855e+01           5     156.2
[eql   16/20]   -8.8188076458  3.179502e+01          10     202.3
[eql   20/20]   -8.8188783480  3.133496e+01          17     248.2

Sampling:

        block           E_avg       E_err         E_block             W         nodes    dt[s/bl]     t[s]
[blk    4/40]   -8.8187568003                 -8.8187568003  3.098012e+01          21     73.694     294.8
[blk    8/40]   -8.8187725343                 -8.8187879416  3.163691e+01          24     11.407     340.4
[blk   12/40]   -8.8187579812   2.046e-05     -8.8187293808  3.186207e+01          27     11.546     386.6
[blk   16/40]   -8.8187594592   2.943e-05     -8.8187639799  3.088913e+01          43     37.973     538.5
[blk   20/40]   -8.8187861840   2.438e-05     -8.8188918461  3.170907e+01          47     21.260     623.5
[blk   24/40]   

In [12]:
be_mps, be_msd = np.asarray(be_mps), np.asarray(be_msd)
N_EQ = params.n_eql_blocks + 1

# print(f"E_exact (ED)                      {e_exact:.12f}")
print(f"<psi_T|H|psi_T>  (DMRG trial)    {e_T:.12f}")
print(f"CPMC, DMRG trial via MPS-MPS     {float(mean_mps):.12f} +- {float(err_mps):.2e}")
print(f"CPMC, DMRG trial via MSD sum     {float(mean_msd):.12f} +- {float(err_msd):.2e}")
print(f"mean difference between routes   {abs(float(mean_mps) - float(mean_msd)):.3e}")

d = np.abs(be_mps - be_msd)
print(f"\n|dE| per block: max {d.max():.3e}   median {np.median(d):.3e}   final {d[-1]:.3e}")
print("  first 12: " + "  ".join(f"{x:.0e}" for x in d[:12]))
print("  last  12: " + "  ".join(f"{x:.0e}" for x in d[-12:]))

<psi_T|H|psi_T>  (DMRG trial)    -8.818756381246
CPMC, DMRG trial via MPS-MPS     -8.818747521183 +- 1.82e-05
CPMC, DMRG trial via MSD sum     -8.818801964153 +- 1.71e-05
mean difference between routes   5.444e-05

|dE| per block: max 4.101e-04   median 9.182e-05   final 9.632e-05
  first 12: 9e-05  7e-05  1e-04  2e-04  1e-04  4e-04  3e-04  7e-05  1e-04  1e-04  2e-04  4e-04
  last  12: 8e-05  1e-05  1e-05  5e-05  8e-05  1e-04  2e-04  3e-04  4e-04  9e-05  1e-04  1e-04


In [13]:
"""PERFECT SAMPLING FROM THE DMRG TRIAL.  Est runtime 20 s.

Draws whole determinants |d> = |n_a, n_b> with probability exactly

    p(d) = |<d|Psi_T>|^2 / <Psi_T|Psi_T>

in one left-to-right pass: no Markov chain, no burn-in, no autocorrelation, no
rejections. Ferris & Vidal, PRB 85, 165146 (2012); tensornetwork.org's writeup;
the algorithm ITensorMPS calls `sample!`.

HOW IT WORKS. Factor p(d) = p(l_0) p(l_1|l_0) ... over the local indices
l = n_a + 2 n_b and sample the conditionals in order. Each one is a one-site
reduced density matrix with the sites to the LEFT projected onto what has
already been drawn and the sites to the RIGHT traced out. Tracing out the right
half is free if the MPS is right-canonical, sum_l B[:,l,:] B[:,l,:]^T = I,
because then everything beyond the current site contracts to the identity. With
a normalised right-canonical MPS and a running left vector v (||v|| = 1),

    w[l, :] = v . A_x[:, l, :]        p(l | l_<x) = ||w[l]||^2

and sum_l ||w[l]||^2 = ||v||^2 = 1 exactly, so the conditional is normalised by
construction. Draw l, set v <- w[l]/||w[l]||, step right. One determinant costs
O(L d chi^2) -- the same as one overlap.

NO CHANNEL FACTORISATION HERE, and that is the whole difference from
`mps_trial_cpmc.ipynb`. There the trial was a single determinant, a spin
product, so p(n_a, n_b) = p_a(n_a) p_b(n_b) and the two spin channels could be
sampled independently at chi_sigma each. A DMRG state is correlated and that
fails outright. Measured on this chi=8 trial (see
`dmrg_trial_cpmc_conventions.md`):

    spin-Schmidt rank of Psi_T ......................... 61  (1 <=> spin product)
    TV( p(n_a, n_b) , p_a(n_a) p_b(n_b) ) .............. 0.59

Sampling the marginals separately would therefore draw from a distribution that
differs from the true one on 59% of its mass. So everything here runs on the
d = 4 MPS directly, at the full bond dimension, and the sampler's `d` is 4
rather than 2. That is the only change the code needs -- nothing below is
specialised to d = 2 -- but it is a change of object, not of constant.

The sampler was checked against the exact 4900-determinant distribution that M
provides -- chi^2 = 2791 on 2839 dof -- and that check is recorded in
`dmrg_trial_cpmc_conventions.md` rather than re-run here.
"""
from collections import namedtuple

Sample = namedtuple("Sample", "config logp amp")


# ---------------------------------------------------------------- canonical form
def right_canonicalize(ts, normalize=True):
    """Right-to-left sweep to right-canonical form, orthogonality centre on site 0.

    sum_l B_x[:, l, :] B_x[:, l, :]^T = I for every x >= 1 -- the property that
    makes the sampling conditionals normalised by construction.

    Done as an LQ, i.e. a QR of the transpose: with M = A_x.reshape(Dl, d*Dr)
    and M^T = Q R, the right factor is Q^T (orthonormal ROWS, so right
    isometric) and R^T is pushed one site left. Bond dimensions can only shrink,
    to min(Dl, d*Dr), and they are python ints, so shapes stay static.

    Returns (tensors, norm). Run it ONCE, then draw as many samples as you like.
    """
    ts = [jnp.asarray(t) for t in ts]
    for x in range(len(ts) - 1, 0, -1):
        Dl, d, Dr = ts[x].shape
        q, r = jnp.linalg.qr(ts[x].reshape(Dl, d * Dr).conj().T, mode="reduced")
        ts[x] = q.conj().T.reshape(-1, d, Dr)
        ts[x - 1] = jnp.tensordot(ts[x - 1], r.conj().T, axes=([2], [0]))
    norm = jnp.linalg.norm(ts[0])
    if normalize:
        ts[0] = ts[0] / jnp.where(norm == 0.0, 1.0, norm)
    return ts, norm


def canonical_error(ts):
    """max_x || sum_l B_x B_x^T - I ||_inf over x >= 1. Zero iff right-canonical."""
    return max(float(jnp.abs(jnp.einsum("axc,bxc->ab", t, t.conj())
                             - jnp.eye(t.shape[0])).max()) for t in ts[1:])


# --------------------------------------------------------------------- the draw
def perfect_sample(ts, key=None, us=None, eps=1e-300):
    """One perfect sample from a NORMALISED RIGHT-CANONICAL MPS.

    Returns (config, logp, amp): the local indices l_x, log p(config), and
    <config|psi> of the normalised state, sign included. |amp|^2 == exp(logp).

    Randomness enters only as L uniforms, either from `key` or supplied as `us`
    -- the same pattern as the fast sweep, so a sampled step can be driven off
    the same RNG stream as a propagation step.

    The site index is an unrolled inverse CDF (d-1 comparisons summed) rather
    than `jnp.searchsorted`, which XLA cannot flatten at d = 4, and rather than
    `jax.random.categorical`, which would take the log of a probability that is
    legitimately zero. Worth 1.5x at this size. `logp` is not accumulated
    separately: the state is normalised, so log p = 2 log|amp| identically.
    """
    n = len(ts)
    us = jax.random.uniform(key, (n,)) if us is None else us
    v = jnp.ones((1,), ts[0].dtype)
    cfg, logamp = [], jnp.zeros(())
    for x in range(n):
        w = jnp.tensordot(v, ts[x], axes=([0], [0]))       # (d, Dr)
        p = jnp.einsum("sr,sr->s", w, w.conj()).real       # sums to ||v||^2 = 1
        c = jnp.cumsum(p)
        u = us[x] * c[-1]
        s = sum((u >= c[k]).astype(jnp.int32) for k in range(p.shape[0] - 1))
        nrm = jnp.sqrt(jnp.maximum(p[s], eps))
        v = w[s] / nrm
        cfg.append(s)
        logamp = logamp + jnp.log(nrm)
    return Sample(jnp.stack(cfg).astype(jnp.int32), 2.0 * logamp,
                  jnp.exp(logamp) * v[0])


def perfect_sample_batch(ts, key, n_samples):
    """`n_samples` independent perfect samples, vmapped over split keys."""
    return jax.vmap(perfect_sample, in_axes=(None, 0))(ts, jax.random.split(key, n_samples))


def sample_mps(ts, key, n_samples=None):
    """Canonicalise, normalise, then draw. Returns (samples, norm)."""
    ts, norm = right_canonicalize(ts)
    s = perfect_sample(ts, key) if n_samples is None else perfect_sample_batch(ts, key, n_samples)
    return s, norm


def mps_amplitude(ts, config):
    """<config|psi>. Contracts all d and then picks, so the shared tensor stays a
    GEMM: the obvious `v @ A[:, config[x], :]` gathers a whole chi x chi matrix
    per sample and goes memory bound under vmap (7-38x slower, measured in the
    companion notebook)."""
    v = jnp.ones((1,), ts[0].dtype)
    for x, A in enumerate(ts):
        v = jnp.tensordot(v, A, axes=([0], [0]))[config[x]]
    return v[0]


def mps_amplitudes(ts, configs):
    return jax.vmap(mps_amplitude, in_axes=(None, 0))(ts, configs)


# ------------------------------------------------- configurations <-> determinants
_CODE_A = {int(o @ (1 << np.arange(L))): i for i, o in enumerate(OCCA)}


def config_to_strings(cfg):
    """(L,) local indices l = n_a + 2 n_b  ->  (alpha string index, beta string index).

    Vectorised over a batch. Raises if a draw has the wrong electron count, which
    cannot happen: the MPS carries exact U(1) x U(1) quantum numbers, so every
    other configuration has amplitude identically zero.
    """
    cfg = np.asarray(cfg)
    na, nb = cfg % 2, cfg // 2
    pw = 1 << np.arange(L)
    ca, cb = na @ pw, nb @ pw
    return (np.array([_CODE_A[int(x)] for x in ca]),
            np.array([_CODE_A[int(x)] for x in cb]))

In [14]:
"""CPMC ON AN MSD TRIAL SELECTED BY SAMPLING THE DMRG STATE.  Est runtime 2 min.

Same experiment as in `mps_trial_cpmc.ipynb`, on a correlated trial. Draw
determinants from |Psi_T>, keep the ones that came up with their EXACT
coefficients, and run CPMC on that truncated expansion -- deterministically,
same seed, same propagator, same two kernels as the full MSD route above. The
only stochastic ingredient is which determinants made the list.

    |Psi_tilde> = sum_{d in sampled} c_d |d>,    c_d = M[a, b]

Concretely that is a MASK on M: `Mt = M * mask` and `MHt = contract_2e(h2e, Mt)`
(H applied to the TRUNCATED vector, not a truncation of MH -- H connects
determinants outside the sampled set and those terms are part of
<Psi_tilde|H|phi>). Everything else is byte for byte the machinery already in
this notebook, so the sampled run costs exactly what the full MSD run costs:
still 140 small determinants and a 70x70 bilinear form per walker, however
sparse the mask is.

WHAT CONVERGENCE LOOKS LIKE HERE, AND WHY IT IS NOT THE HF STORY. In the
companion notebook the trial was a single determinant, a spin product, and the
sampled support could be taken as a PRODUCT of two channel sets: 70 + 70
configurations to collect instead of 4900 pairs, so ~1.5e6 draws made the trial
EXACT and CPMC reproduced the reference to the last digit.

None of that is available here. The DMRG state has spin-Schmidt rank 61, so
there is no product structure to exploit and the support is the 4900 pairs
themselves. Collecting all of them is hopeless -- and pointless, because the
weight is what matters, not the count:

    fidelity  <Psi_tilde|Psi_T>^2 / <Psi_tilde|Psi_tilde> = sum_{d in S} |c_d|^2

and sampling visits determinants in proportion to exactly that. So the trial
converges in FIDELITY fast while the support is still far from complete, which
is the useful behaviour: it is determinant selection by importance, the same
idea as a stochastically selected CI expansion. The table below is the honest
statement of it -- no machine-precision agreement, but a controlled and rapidly
shrinking bias, and the CPMC energy tracks the fidelity rather than the count.
"""
N_SAMPLES  = 4_000_000       # <-- the knob
N_SMALL    = 2_000           # a deliberately starved trial, for contrast
KEY_MSD    = jax.random.PRNGKey(1)
CHUNK      = 500_000


def sample_support(ts, n_samples, key, chunk=CHUNK):
    """Draw n_samples determinants and return the (NS, NS) boolean support mask.

    Batched so n_samples can be large without holding every draw: only the mask
    survives a batch. Also returns the hit counts, which are the empirical
    p_hat and are worth looking at next to the exact |c_d|^2.
    """
    can, _ = right_canonicalize(ts)
    mask = np.zeros((NS, NS), bool)
    hits = np.zeros((NS, NS), np.int64)
    left, k = int(n_samples), key
    f = jax.jit(perfect_sample_batch, static_argnums=2)
    while left > 0:
        k, sub = jax.random.split(k)
        m = min(chunk, left); left -= m
        ia, ib = config_to_strings(f(can, sub, m).config)
        np.add.at(hits, (ia, ib), 1)
    mask = hits > 0
    return mask, hits


def make_sampled_msd(mask):
    """(overlap, energy) closures for the masked trial, in the notebook's signature."""
    Mt = M * mask
    MHt = apply_H(Mt)
    Mtj, MHtj = jnp.asarray(Mt), jnp.asarray(MHt)

    def overlap(walker, trial_data=None):
        ca, cb = walker
        return _dets(ca) @ Mtj @ _dets(cb)

    def energy(walker, ham_data=None, meas_ctx=None, trial_data=None):
        ca, cb = walker
        da, db = _dets(ca), _dets(cb)
        return (da @ MHtj @ db) / (da @ Mtj @ db)

    return overlap, energy, Mt, MHt


# ------------------------------------------------------------ reference walkers
_rng = np.random.default_rng(7)
_W = (jnp.asarray(np.stack([Ca + 0.25 * _rng.standard_normal(Ca.shape) for _ in range(32)])),
      jnp.asarray(np.stack([Cb + 0.25 * _rng.standard_normal(Cb.shape) for _ in range(32)])))
_ov_ref = np.asarray(jax.jit(jax.vmap(msd_overlap))(_W))
_e_ref = np.asarray(jax.jit(jax.vmap(msd_energy))(_W))

# ------------------------------------------------------- how the trial converges
print(f"{'N drawn':>10s} {'dets':>6s} {'of 4900':>8s} {'fidelity':>11s} {'1-fidelity':>11s} "
      f"{'<H>_trial':>13s} {'max ov err':>11s} {'max E err':>10s}")
_rows = []
for _n in (2_000, 20_000, 200_000, 2_000_000):
    _m, _ = sample_support(ket_T, _n, KEY_MSD)
    _ov_f, _e_f, _Mt, _MHt = make_sampled_msd(_m)
    _w = float((_Mt ** 2).sum())
    _et = float((_Mt * _MHt).sum() / (_Mt * _Mt).sum())
    _ov = np.asarray(jax.jit(jax.vmap(_ov_f))(_W)); _e = np.asarray(jax.jit(jax.vmap(_e_f))(_W))
    print(f"{_n:10d} {int(_m.sum()):6d} {_m.sum()/(NS*NS):8.3f} {_w:11.8f} {1-_w:11.2e} "
          f"{_et:13.9f} {np.abs(_ov/_ov_ref - 1).max():11.2e} "
          f"{np.abs(_e - _e_ref).max():10.2e}")

# ------------------------------------------------------------------- the trials
_t0 = time.perf_counter()
mask_big, hits_big = sample_support(ket_T, N_SAMPLES, KEY_MSD)
_t_s = time.perf_counter() - _t0
overlap_smsd, energy_smsd, M_big, MH_big = make_sampled_msd(mask_big)
mask_sml, _ = sample_support(ket_T, N_SMALL, KEY_MSD)
overlap_ssml, energy_ssml, M_sml, _ = make_sampled_msd(mask_sml)
_w_big, _w_sml = float((M_big**2).sum()), float((M_sml**2).sum())

print(f"\n{N_SAMPLES} determinants drawn in {_t_s:.1f} s")
print(f"  kept {int(mask_big.sum())} of {NS*NS} determinants "
      f"({int((AMP**2 > 1e-20).sum())} have |c| > 1e-10 at all)")
print(f"  fidelity {_w_big:.9f}   missing weight {1-_w_big:.2e}")
print(f"  trial energy <H>  {float((M_big*MH_big).sum()/(M_big*M_big).sum()):.12f}"
      f"   (full trial {e_T:.12f})")
print(f"  rarest kept, p_hat {hits_big[mask_big].min()/N_SAMPLES:.2e}"
      f"   exact smallest kept |c|^2 {float((AMP**2)[mask_big].min()):.2e}")

# ------------------------------------------------------------------- the runs
mean_smsd, err_smsd, be_smsd, _ = run(
    overlap_smsd, energy_smsd, make_fast_prop_ops(ham, sys_.walker_kind, overlap_smsd))
mean_ssml, err_ssml, be_ssml, _ = run(
    overlap_ssml, energy_ssml, make_fast_prop_ops(ham, sys_.walker_kind, overlap_ssml))
be_smsd, be_ssml = np.asarray(be_smsd), np.asarray(be_ssml)

print(f"\nE_exact (ED)                                {e_exact:.12f}")
print(f"CPMC, full MSD (4900 determinants)         {float(mean_msd):.12f} +- {float(err_msd):.2e}")
print(f"CPMC, MPS-MPS                              {float(mean_mps):.12f} +- {float(err_mps):.2e}")
print(f"CPMC, sampled MSD, {N_SAMPLES} draws "
      f"({int(mask_big.sum())} dets) {float(mean_smsd):.12f} +- {float(err_smsd):.2e}")
print(f"CPMC, sampled MSD, {N_SMALL} draws "
      f"({int(mask_sml.sum())} dets)      {float(mean_ssml):.12f} +- {float(err_ssml):.2e}")
_d = np.abs(be_smsd - be_msd)
print(f"\nmean difference vs the full MSD run:")
print(f"  {N_SAMPLES} draws, fidelity {_w_big:.6f}: "
      f"{abs(float(mean_smsd)-float(mean_msd)):.3e}   per block max {_d.max():.3e}")
print(f"  {N_SMALL} draws, fidelity {_w_sml:.6f}: "
      f"{abs(float(mean_ssml)-float(mean_msd)):.3e}   per block max "
      f"{np.abs(be_ssml-be_msd).max():.3e}")
print(f"\nstatistical error of the run itself: {float(err_msd):.2e}"
      f"  <- the {N_SAMPLES}-draw bias is well inside it")

   N drawn   dets  of 4900    fidelity  1-fidelity     <H>_trial  max ov err  max E err
      2000   1854    0.000  0.12712444    8.73e-01  -0.990280867    1.16e+00   2.48e+01
     20000  14881    0.000  0.36174785    6.38e-01  -3.350902385    1.75e+00   7.53e+01
    200000  94835    0.001  0.64496246    3.55e-01  -5.662285452    2.91e+00   1.97e+01
   2000000 464384    0.003  0.85373447    1.46e-01  -7.353962765    3.24e+00   9.08e+00

4000000 determinants drawn in 9.0 s
  kept 710644 of 165636900 determinants (116516629 have |c| > 1e-10 at all)
  fidelity 0.894304104   missing weight 1.06e-01
  trial energy <H>  -7.711666879804   (full trial -8.818756381246)
  rarest kept, p_hat 2.50e-07   exact smallest kept |c|^2 2.05e-13

Equilibration:

        block           E_blk             W        nodes      t[s]
[eql    0/20]  -10.8992630197  3.200000e+01           0       0.0
[eql    4/20]   -9.0344388553  2.871269e+01           0      74.3
[eql    8/20]   -8.8446410934  2.826252e+01     

KeyboardInterrupt: 